# Train a model on a live OpenRange world with TRL GRPO + LoRA

OpenRange admits **content-addressed** agent-training worlds. TRL's `GRPOTrainer`
optimizes a policy from the *spread* of rewards across a group of rollouts. This
notebook wires the two: a small instruct model learns to fix a real bug in a live
software world, graded by a held-out test suite it never sees.

The gym tracks the policy. Every rollout realizes a fresh workspace, the model
acts through file tools, and the final tree is graded — there is no static dataset
of "correct" trajectories. The reward is the world's own verdict.

It runs on a laptop (MPS or CUDA). The point is the **mechanics and a reward you
can see is real** — §3 makes the gradient concrete *before* any GPU work — not a
SOTA run.

For the *offensive* cyber surface — exploiting a live web service over HTTP,
with an in-place curriculum — see `trl_grpo_cyber.ipynb`.

## 1. Install

```bash
uv pip install "openrange-trl[train]"   # torch, transformers, trl, datasets, accelerate, peft
```

`import openrange` never needs this extra — only the live training path below does.

## 2. Admit a world

`admit` turns a pack manifest into a **content-addressed snapshot**: a frozen,
reproducible world. We use `calc_sum`, a one-line bug fix — one failing test, one
passing test, graded by a held-out suite. The `snapshot_id` (a sha256) tags
everything downstream, so every trajectory is provenanced to the exact world.

In [1]:
from swe import SwePack

from openrange.core.admit import AdmissionFailure, admit

pack = SwePack()
snapshot = admit(pack, manifest={"instance": "calc_sum"}, max_repairs=0)
assert not isinstance(snapshot, AdmissionFailure), snapshot
snapshot.snapshot_id

'sha256:a051dbdf65f58a70e9509e72f76944595bee6ff2d308325b77b9c305f880ca83'

## 3. See the reward before you train

The reward *is* the world's held-out verdict: `1.0` when every test passes, else
the fraction of subgoals. `calc_sum` is a trap worth understanding — both `add`
and `subtract` read `return a - b`, but only `add` is buggy (it should sum).

Driving the env's own tools by hand maps the whole reward surface. Each call below
resets a fresh workspace, makes one edit, and grades it — no model, no GPU.

In [2]:
from openrange_trl import env_trajectory, make_environment_factory

probe_factory = make_environment_factory(pack, [snapshot], "or-runs/notebook/probe")


def grade(label, edit):
    env = probe_factory()
    env.reset()
    edit(env)
    traj = env_trajectory(env)
    env.service.close()
    print(f"{label:<26} reward={traj.reward.scalar:.1f}  resolved={traj.success}")


grade("no-op (read only)", lambda e: e.read_file("calc/core.py"))
grade(
    "naive replace-all",
    lambda e: e.apply_patch("calc/core.py", "return a - b", "return a + b"),
)
grade(
    "targeted fix (add only)",
    lambda e: e.apply_patch(
        "calc/core.py",
        "def add(a, b):\n    return a - b",
        "def add(a, b):\n    return a + b",
    ),
)
grade(
    "break subtract",
    lambda e: e.apply_patch(
        "calc/core.py",
        "def subtract(a, b):\n    return a - b",
        "def subtract(a, b):\n    return a * b",
    ),
)

no-op (read only)          reward=0.5  resolved=False


naive replace-all          reward=0.5  resolved=False


targeted fix (add only)    reward=1.0  resolved=True


break subtract             reward=0.0  resolved=False


Four edits, three distinct grades:

| edit | reward | why |
|------|--------|-----|
| no-op | **0.5** | the failing test stays red, the passing one is free |
| naive replace-all | **0.5** | greens `add` but **breaks `subtract`** — back to the floor |
| targeted fix (add only) | **1.0** | both tests pass |
| break subtract | **0.0** | both tests red |

That `0.5 → 1.0` gap is the **spread GRPO turns into a gradient**, and the naive
replace-all is the local trap a weak policy gets stuck in. Training is teaching
the policy to find the *targeted* edit. Everything below is how a model chases
that signal on its own. (This surface is asserted in
`tests/test_trl_adapter.py::TestRewardSpread`, so it can't silently drift.)

## 4. Wrap the world as a TRL environment

The torch-free adapter `openrange_trl` exposes the three seams TRL needs:

- **`build_grpo_dataset`** — the prompt rows (problem statement, `snapshot_id`-tagged).
- **`make_environment_factory`** — builds one `OpenRangeEnv` per rollout (used above
  for the probe). Each env resets a fresh workspace and exposes five **tools**:
  `read_file`, `write_file`, `list_dir`, `apply_patch`, `run_tests`.
- **`make_reward_func`** — grades the final workspace with the held-out suite and
  shapes it into the scalar you just saw.

GRPO never sees the grader; it only reads the reward the world returns.

In [3]:
from datasets import Dataset
from openrange_trl import build_grpo_dataset, make_reward_func

dataset = Dataset.from_list(build_grpo_dataset(snapshot, repeat=4))
environment_factory = make_environment_factory(
    pack, [snapshot], "or-runs/notebook/envs"
)
reward_func = make_reward_func()

## 5. Load the policy + a LoRA adapter

We train **LoRA adapters** on a small instruct model, not the full weights — the
Unsloth-style recipe. The base stays frozen (so it fits a laptop's memory); GRPO
updates only the low-rank adapters on the attention projections. Bump `model_name`
to a larger instruct model for rollouts strong enough to clear the 0.5 trap.

In [4]:
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 6688.63it/s]

## 6. Configure GRPO

A *group* is one optimizer step's worth of rollouts. With
`per_device_train_batch_size == num_generations` and `steps_per_generation=1`,
each step samples `num_generations` completions of the **same** prompt, grades
them, and turns their reward *spread* into advantages. `beta=0` drops the
reference model (lighter); `use_vllm=False` generates through transformers.

This is sized to run **one real step in a few laptop minutes**. Scale
`num_generations`, `max_steps`, and `temperature` up for a real run.

In [5]:
from trl import GRPOConfig

num_generations = 4
config = GRPOConfig(
    output_dir="or-runs/notebook/trainer",
    per_device_train_batch_size=num_generations,
    num_generations=num_generations,
    steps_per_generation=1,
    gradient_accumulation_steps=1,
    max_steps=1,
    learning_rate=1e-4,
    beta=0.0,
    temperature=1.0,
    max_completion_length=256,
    max_tool_calling_iterations=4,
    use_vllm=False,
    log_completions=False,
    logging_steps=1,
    report_to="none",
    disable_tqdm=True,
    save_strategy="no",
    bf16=False,
    fp16=False,
)

## 7. Train

`GRPOTrainer` discovers the env's tool methods by reflection and hands them to the
chat template, runs the multi-turn rollouts, reads `reward_func`, and steps the
LoRA adapters.

A 0.5B policy with four rollouts will usually **floor at 0.5** — it lands in the
naive-patch trap from §3, so `reward_std` is often `0` here. That is the honest
starting point, not a failure: the gradient appears once a rollout diverges (a
stronger model, higher temperature, or more samples), reaching exactly the
`1.0`/`0.0` grades §3 proved are on the table.

In [6]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_func],
    args=config,
    train_dataset=dataset,
    processing_class=tokenizer,
    environment_factory=environment_factory,
    peft_config=lora,
)
trainer.train()

{'loss': '0', 'grad_norm': '0', 'learning_rate': '0.0001', 'num_tokens': '4236', 'completions/mean_length': '159', 'completions/min_length': '37', 'completions/max_length': '256', 'completions/clipped_ratio': '0.5', 'completions/mean_terminated_length': '92', 'completions/min_terminated_length': '37', 'completions/max_terminated_length': '147', 'tools/call_frequency': '1', 'tools/failure_frequency': '0', 'rewards/reward_func/mean': '0.5', 'rewards/reward_func/std': '0', 'reward': '0.5', 'reward_std': '0', 'frac_reward_zero_std': '1', 'entropy': '3.245', 'clip_ratio/low_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/high_max': '0', 'clip_ratio/region_mean': '0', 'step_time': '83.08', 'epoch': '0.25'}
{'train_runtime': '84.25', 'train_samples_per_second': '0.047', 'train_steps_per_second': '0.012', 'train_loss': '0', 'epoch': '0.25'}


TrainOutput(global_step=1, training_loss=0.0, metrics={'train_runtime': 84.2522, 'train_samples_per_second': 0.047, 'train_steps_per_second': 0.012, 'train_loss': 0.0, 'epoch': 0.25})

## 8. Inspect the learning signal

The envs `GRPOTrainer` built hold the last batch's graded episodes. We read each
rollout's reward and whether it *resolved* the world (held-out suite fully green),
then export a `snapshot_id`-tagged trajectory — the seam any downstream consumer
(replay, fine-tuning, analysis) reads.

In [7]:
envs = list(trainer.environments or ())
rewards = [env.reward for env in envs]
resolved = sum(1 for env in envs if env.report is not None and env.report.passed)
spread = max(rewards) - min(rewards) if rewards else 0.0

print(f"rewards : {[round(r, 3) for r in rewards]}")
print(f"spread  : {spread:.3f}   (non-zero => a real GRPO gradient)")
print(f"resolved: {resolved}/{len(envs)}")

trajectory = env_trajectory(envs[0])
print(f"world   : {trajectory.snapshot_id}")
print(f"turns   : {len(trajectory.steps)}   success: {trajectory.success}")

for env in envs:
    env.service.close()

rewards : [0.5, 0.5, 0.5, 0.5]
spread  : 0.000   (non-zero => a real GRPO gradient)
resolved: 0/4
world   : sha256:a051dbdf65f58a70e9509e72f76944595bee6ff2d308325b77b9c305f880ca83
turns   : 0   success: False


## Where this goes

You just saw the reward is real (§3) and the loop runs end-to-end (§7–8). Two
seams carry it further:

- **Curriculum.** `reward_variance_policy` reads a round's reward spread; when it
  collapses, `auto_evolve` mutates the world in place — or, for the SWE pack, steps
  an instance ladder (`calc_sum → shapes_area → notes_app`). The gym tracks the
  policy, live.
- **Provenance.** Every trajectory is tagged with the `snapshot_id`, so an evolved
  curriculum stays fully attributable.

The deterministic, torch-free seam and its reward-dynamics tests (the §3 surface,
asserted) live in the `openrange-trl` package + `tests/test_trl_adapter.py`,
and the gated live mechanics test is `tests/test_trl_live.py`.